In [0]:
%sql
-- Cria o schema (dataset)
CREATE SCHEMA IF NOT EXISTS workspace.nyc_taxi

In [0]:
%sql
-- Cria os volumes de armazenamento dos dados brutos
CREATE VOLUME IF NOT EXISTS workspace.nyc_taxi.landing_zone;

In [0]:
# Os arquivos foram inseridos na landing zone manualmente
# Verifica os arquivos na landing zone
display(dbutils.fs.ls("/Volumes/workspace/nyc_taxi/landing_zone/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/nyc_taxi/landing_zone/yellow_tripdata_2023-01.parquet,yellow_tripdata_2023-01.parquet,47673370,1778385211000
dbfs:/Volumes/workspace/nyc_taxi/landing_zone/yellow_tripdata_2023-02.parquet,yellow_tripdata_2023-02.parquet,47748012,1778385211000
dbfs:/Volumes/workspace/nyc_taxi/landing_zone/yellow_tripdata_2023-03.parquet,yellow_tripdata_2023-03.parquet,56127762,1778385213000
dbfs:/Volumes/workspace/nyc_taxi/landing_zone/yellow_tripdata_2023-04.parquet,yellow_tripdata_2023-04.parquet,54222699,1778385213000
dbfs:/Volumes/workspace/nyc_taxi/landing_zone/yellow_tripdata_2023-05.parquet,yellow_tripdata_2023-05.parquet,58654627,1778385214000


In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as f
from pyspark.sql import types as t

# Lê os arquivos parquet da landing zone e define o schema explicitamente das colunas necessárias para que não haja inconsistências nos tipos dos campos

LANDING_PATH = "/Volumes/workspace/nyc_taxi/landing_zone/"

df_final = None

for month in ["01", "02", "03", "04", "05"]:
    path = f"{LANDING_PATH}yellow_tripdata_2023-{month}.parquet"
    df = spark.read.parquet(path)
    df = df.select(
        f.col("VendorID").cast(t.LongType()).alias("vendor_id"),
        f.col("passenger_count").cast(t.IntegerType()),
        f.col("total_amount").cast(t.DoubleType()),
        f.col("tpep_pickup_datetime").cast(t.TimestampType()),
        f.col("tpep_dropoff_datetime").cast(t.TimestampType()),
    )

    if df_final is None:
        df_final = df
    else:
        df_final = df_final.union(df)

print(f"Total de linhas: {df_final.count():,}")
df_final.printSchema()

Total de linhas: 16,186,386
root
 |-- vendor_id: long (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)



In [0]:
# Salva os arquivos como Delta Table no catálogo
df_final.write.format("delta").mode("overwrite").saveAsTable("workspace.nyc_taxi.yellow_trips")

In [0]:
%sql
-- Consultando a tabela para validacão
SELECT
  *
FROM
  workspace.nyc_taxi.yellow_trips
LIMIT 10

VendorID,passenger_count,total_amount,tpep_pickup_datetime,tpep_dropoff_datetime
1,2,39.9,2023-04-01T00:14:49.000Z,2023-04-01T00:45:01.000Z
2,1,81.8,2023-04-01T00:00:24.000Z,2023-04-01T00:56:19.000Z
1,2,18.4,2023-04-01T00:03:50.000Z,2023-04-01T00:14:42.000Z
1,1,16.0,2023-04-01T00:53:18.000Z,2023-04-01T01:01:28.000Z
2,2,17.4,2023-04-01T00:07:00.000Z,2023-04-01T00:17:16.000Z
1,6,17.85,2023-04-01T00:08:59.000Z,2023-04-01T00:15:39.000Z
2,1,61.92,2023-04-01T00:27:52.000Z,2023-04-01T00:43:07.000Z
2,1,33.62,2023-04-01T00:48:38.000Z,2023-04-01T01:08:37.000Z
1,0,48.8,2023-04-01T00:22:28.000Z,2023-04-01T00:34:29.000Z
2,1,17.16,2023-04-01T00:27:06.000Z,2023-04-01T00:34:06.000Z


In [0]:
%sql
-- Adiciona descrição da tabela
ALTER TABLE
  workspace.nyc_taxi.yellow_trips
SET TBLPROPERTIES
  ('comment' = 'Tabela com dados de corridas de táxi em Nova York para o ano de 2023.');

-- Adiciona descrição das colunas
ALTER TABLE
  workspace.nyc_taxi.yellow_trips
ALTER COLUMN
  VendorID
  COMMENT 'Código do fornecedor (1: Creative Mobile Technologies, 2: Curb Mobility, 6: Myle Technologies, 7: Helix).';

ALTER TABLE
  workspace.nyc_taxi.yellow_trips
ALTER COLUMN
  passenger_count
  COMMENT 'Número de passageiros no veículo.';

ALTER TABLE
  workspace.nyc_taxi.yellow_trips
ALTER COLUMN
  total_amount
  COMMENT 'Valor total cobrado do passageiro.';

ALTER TABLE
  workspace.nyc_taxi.yellow_trips
ALTER COLUMN
  tpep_pickup_datetime
  COMMENT 'Data e hora em que o taxímetro foi acionado.';

ALTER TABLE
  workspace.nyc_taxi.yellow_trips
ALTER COLUMN
  tpep_dropoff_datetime
  COMMENT 'Data e hora em que o taxímetro foi desligado.';